# PharmaVigil: SafetyFormer full-scale training (Kaggle GPU)

Trains SafetyFormer on the full FAERS gold tables exported from Databricks
(`data/notebooks/databricks/03_export_for_training`), evaluates it against PRR, ROR, IC and EBGM on the
OMOP and EU-ADR reference sets, and runs the leakage check. Outputs go to `/kaggle/working/m4`; download
them and send them to the planning chat.

*Not for clinical or regulatory use. Spontaneous reports do not prove that a drug caused a reaction.*

**Settings:** Accelerator **GPU T4 x2** (or P100), Internet **on**. Input dataset:
`pharmavigil-faers-gold` (folders `report`, `report_drug`, `report_reaction`).

In [ ]:
GIT_REF = "main"  # use the pull-request branch name until it is merged
SCOPE = "full: FAERS 2004Q1–2026Q2"  # edit to the last quarter actually loaded
GOLD = "/kaggle/input/pharmavigil-faers-gold"
REPO = "https://github.com/sumitparmar19/PharmaVigil.git"


In [ ]:
!pip install -q "pharmavigil-data @ git+{REPO}@{GIT_REF}#subdirectory=data" "pharmavigil-ml[train] @ git+{REPO}@{GIT_REF}#subdirectory=ml"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, importlib.resources as r
os.environ["PV_LAKE"] = "/kaggle/working/lake"
OUTCOME_MAP = str(r.files("pharmavigil_data.resources") / "outcome_pt_map.csv")
!pv-data refsets

## 1. Train and compare on the reference sets (fixed seed 13)

In [ ]:
!pv-ml safetyformer --gold {GOLD} --refsets /kaggle/working/lake/bronze/refsets --outcome-map {OUTCOME_MAP} \
    --out /kaggle/working/m4 --models-dir /kaggle/working/models --tracking-uri file:/kaggle/working/mlruns \
    --scope "{SCOPE}" --steps 200000 --batch-size 1024 --d-model 256 --n-layers 4 --n-heads 8 --min-count 5 --seed 13

## 2. Leakage check: shuffled vs time-ordered split (test window = last full year)

In [ ]:
!pv-ml leakage --gold {GOLD} --out /kaggle/working/m4 --models-dir /kaggle/working/models-leakage \
    --tracking-uri file:/kaggle/working/mlruns --scope "{SCOPE}" --test-start 2025-01-01 \
    --steps 50000 --batch-size 1024 --d-model 256 --n-layers 4

## 3. Package the evidence

In [ ]:
!sha256sum /kaggle/working/models/safetyformer.pt
!cd /kaggle/working && zip -qr pharmavigil_m4_evidence.zip m4 models/vocab.json && ls -la pharmavigil_m4_evidence.zip